## Api ingestion

- ## import the library 

In [0]:
import requests 
import json 
from pyspark.sql.functions import * 
from pyspark.sql.types import *

**create catalog**

In [0]:
# DBTITLE 1,Create Catalog , Schema and Volume

spark.sql("CREATE CATALOG IF NOT EXISTS workspace")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cricket_api_project")


base_path='/Volumes/workspace/default/csv'


Calling cricket Api

In [0]:
###CALLING Cricket API
API_KEY='4e60c588-2294-4d25-9cdf-31705f1fda7c'
api_url=f"https://api.cricapi.com/v1/countries?apikey=4e60c588-2294-4d25-9cdf-31705f1fda7c&offset=0"

response =requests.get(api_url)
response.raise_for_status()

api_data=response.json()
print(api_data.keys())



In [0]:
print(json.dumps(api_data,indent= 2)[:2000])

**SAVE RAW API Response in the Volumes**

In [0]:
raw_file_path=f'{base_path}/current_matches_raw.json'

with open (raw_file_path,'w') as file:
    json.dump(api_data,file)

print("RAW API data is save at the :",raw_file_path)

**create bronze layer**

In [0]:
bronze_data=[{
"source_api":api_url,
"raw_json":json.dumps(api_data),
"ingestion_time":None
}]

In [0]:
bronze_schema=StructType([
StructField("source_api",StringType(),True),
StructField("raw_json",StringType(),True),
StructField("ingestion_time",TimestampType(),True)
])


In [0]:
bronze_df=spark.createDataFrame(bronze_data,bronze_schema)\
    .withColumn("ingestion_time",current_timestamp())

In [0]:
display(bronze_df)

**Create a bronze layer**

In [0]:
bronze_df.write\
    .format('delta')\
    .mode('overwrite')\
    .saveAsTable("workspace.default.cricket_bronze_current_matches")

print("BRONZE TABLE CREATED SUCCESSFULLY")


In [0]:
%sql
select * from workspace.default.cricket_bronze_current_matches 